In [1]:
import logging
from pydantic import BaseModel, Field
from typing import Optional, AsyncGenerator, List, Union

from llama_index.llms.ollama import Ollama
from llama_index.core import PromptTemplate

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    filename="ollama_chat.log",
)
logger = logging.getLogger(__name__)

In [3]:
class ChatMessage(BaseModel):
    role: str = Field(default="user")
    content: str


class ChatResponse(BaseModel):
    response: Optional[Union[str, List[ChatMessage]]] = None
    metadata: dict = Field(default_factory=dict)
    error: Optional[str] = None
    stream: Optional[AsyncGenerator[str, None]] = None

    class Config:
        arbitrary_types_allowed = True

In [6]:
class OllamaChat:
    def __init__(self):
        self.system_prompt = """You are a helpful, respectful and honest assistant. 
        Always provide accurate information and if you're not sure about something, 
        admit it. Follow these rules:
        1. Keep responses clear and concise
        2. Use markdown formatting when appropriate
        3. If asked about coding, provide working examples
        4. Never generate harmful or unethical content
        5. Always maintain a professional tone
        """

        try:
            self.llm = Ollama(
                model="llama3.2:1b",
                temperature=0.7,
                context_window=4096,
                request_timeout=120.0,
            )
            logger.info("Ollama LLM initialized successfully")
        except Exception as e:
            logger.error(f"Error initializing Ollama LLM: {str(e)}")
            self.llm = None

        self.chat_history: List[ChatMessage] = []

    def _create_prompt(self, user_message: str) -> str:
        prompt_template = PromptTemplate(
            template=(
                "System: {system_prompt}\n"
                "Chat History: {chat_history}\n"
                "User: {user_message}\n"
                "Assistant: "
            )
        )

        chat_history_str = "\n".join(
            [f"{msg.role}: {msg.content}" for msg in self.chat_history[-5:]]
        )

        return prompt_template.format(
            system_prompt=self.system_prompt,
            chat_history=chat_history_str,
            user_message=user_message,
        )

    async def chat(self, message: str) -> ChatResponse:
        """
        Process user input and return a streaming response from the assistant.
        """
        if not self.llm:
            logger.error("LLM is not initialized")
            return ChatResponse(
                error="The assistant is currently unavailable. Please try again later."
            )

        try:
            logger.info(f"Received user message: {message}")
            prompt = self._create_prompt(message)

            async def response_generator():
                full_response = ""
                for chunk in self.llm.stream_complete(prompt):
                    if chunk.delta:
                        full_response += chunk.delta
                        yield chunk.delta

                self.chat_history.append(ChatMessage(role="user", content=message))
                self.chat_history.append(
                    ChatMessage(role="assistant", content=full_response)
                )

            logger.info("Initialized streaming response")
            return ChatResponse(stream=response_generator())

        except Exception as e:
            logger.error(f"Error generating response: {str(e)}")
            return ChatResponse(error=f"[ERROR]: {str(e)}")

    def clear_history(self) -> ChatResponse:
        """
        Clear the chat history.
        """
        try:
            self.chat_history = []
            logger.info("Chat history cleared successfully")
            return ChatResponse(response="Chat history cleared.")
        except Exception as e:
            logger.error(f"Error clearing chat history: {str(e)}")
            return ChatResponse(error=f"[ERROR]: {str(e)}")

    def get_chat_history(self) -> ChatResponse:
        """
        Retrieve the chat history as a list of messages.
        """
        try:
            if not self.chat_history:
                return ChatResponse(response=[], metadata={})
            chat_history_list = [
                {"role": msg.role, "content": msg.content} for msg in self.chat_history
            ]
            return ChatResponse(response=chat_history_list, metadata={})
        except Exception as e:
            logger.error(f"Error retrieving chat history: {str(e)}")
            return ChatResponse(error=f"[ERROR]: {str(e)}")


In [15]:
chat = OllamaChat() 
message = "What is the capital of France?"
response = await chat.chat(message)

In [19]:
print(response.metadata)

{'model': 'llama3.2:1b', 'timestamp': ''}


In [9]:
from llama_index.llms.ollama import Ollama

In [10]:
model = "llama3.2:1b"

llm = Ollama(model=model)
llm

Ollama(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x00000297AE3ECE10>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x00000297CA2DA160>, completion_to_prompt=<function default_completion_to_prompt at 0x00000297CA39D8A0>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, base_url='http://localhost:11434', model='llama3.2:1b', temperature=0.75, context_window=3900, request_timeout=30.0, prompt_key='prompt', json_mode=False, additional_kwargs={}, is_function_calling_model=True, keep_alive=None)

In [11]:
response = llm.stream_complete("Who is Paul Graham?")

for r in response:
    print(r.delta, end="")

Paul Graham is a Canadian-American entrepreneur, programmer, writer, and venture capitalist. He is best known for co-founding Y Combinator, a prominent startup accelerator, as well as his work on the programming language Ruby.

Graham was born in 1961 in Toronto, Canada. He graduated from Harvard University with a degree in computer science and started his career in software development at Apple. In the late 1980s, he founded a company called CodeWheel, which developed a tool for generating HTML code.

In 1995, Graham co-founded Y Combinator (YC) with Jeff Skoll, Peter Thiel, and Sean Parker. The accelerator provided funding to early-stage startups in exchange for equity. Under Graham's leadership, YC has invested in numerous successful companies, including Dropbox, Instagram, Airbnb, and Reddit.

Graham is also known for his writing, particularly on topics related to programming and entrepreneurship. He has written several books, including "The Road to Reinvention" and "How Great Lead